# Step 1: Data Preparation

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('online_retail_dataset.csv')

In [3]:
df.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
43817,493216,22034,ROBIN CHRISTMAS CARD,12,22-12-2009 13:29,0.42,NaN,United Kingdom


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1048575 non-null  object 
 1   StockCode    1048575 non-null  object 
 2   Description  1044203 non-null  object 
 3   Quantity     1048575 non-null  int64  
 4   InvoiceDate  1048575 non-null  object 
 5   Price        1048575 non-null  float64
 6   Customer ID  811893 non-null   float64
 7   Country      1048575 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 64.0+ MB


## Remove Rows without Customer ID

In [5]:
df=df.dropna(subset=['Customer ID'])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 811893 entries, 0 to 1048574
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      811893 non-null  object 
 1   StockCode    811893 non-null  object 
 2   Description  811893 non-null  object 
 3   Quantity     811893 non-null  int64  
 4   InvoiceDate  811893 non-null  object 
 5   Price        811893 non-null  float64
 6   Customer ID  811893 non-null  float64
 7   Country      811893 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 55.7+ MB


## Filter Out Cancellations (Invoice Number Starting with 'C')

In [7]:
df=df[~df['Invoice'].astype(str).str.startswith('C')]

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 793380 entries, 0 to 1048574
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      793380 non-null  object 
 1   StockCode    793380 non-null  object 
 2   Description  793380 non-null  object 
 3   Quantity     793380 non-null  int64  
 4   InvoiceDate  793380 non-null  object 
 5   Price        793380 non-null  float64
 6   Customer ID  793380 non-null  float64
 7   Country      793380 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 54.5+ MB


## Creating Total Price Column

In [9]:
df['TotalPrice']=df['Quantity']* df['Price']

In [10]:
df.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
649330,546917,22138,BAKING SET 9 PIECE RETROSPOT,3,18-03-2011 09:17,4.95,14911.0,EIRE,14.85


## Convert InvoiceDate to datetime objects

In [11]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True)

# Step 2 : Cohort Analysis

## Assign Acquisition Month

In [12]:
df['OrderMonth']=df['InvoiceDate'].dt.to_period('M')

In [13]:
df['CohortMonth']=df.groupby('Customer ID')['OrderMonth'].transform('min')

In [14]:
df.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,OrderMonth,CohortMonth
832058,563808,POST,POSTAGE,2,2011-08-19 11:46:00,18.0,12626.0,Germany,36.0,2011-08,2011-01


## Calculating Cohort Index

### calculating the difference in months.

In [15]:
df['CohortIndex'] = (
    (df['OrderMonth'].dt.year - df['CohortMonth'].dt.year) * 12 +
    (df['OrderMonth'].dt.month - df['CohortMonth'].dt.month)
) + 1

In [16]:
df.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,OrderMonth,CohortMonth,CohortIndex
1032326,579161,47590B,PINK HAPPY BIRTHDAY BUNTING,1,2011-11-28 13:56:00,5.45,17379.0,United Kingdom,5.45,2011-11,2011-11,1


## Creating the Retention Matrix

In [17]:
cohort_data=df.groupby(['CohortMonth','CohortIndex'])['Customer ID'].nunique().reset_index()

In [18]:
cohort_pivot=cohort_data.pivot(index='CohortMonth',columns='CohortIndex',values='Customer ID')

In [19]:
retention_matrix=cohort_pivot.divide(cohort_pivot.iloc[:,0],axis=0)

In [20]:
cohort_pivot

CohortIndex,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,25
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2009-12,955.0,337.0,319.0,406.0,363.0,343.0,360.0,327.0,321.0,346.0,...,289.0,251.0,289.0,270.0,248.0,244.0,301.0,291.0,389.0,69.0
2010-01,383.0,79.0,119.0,117.0,101.0,115.0,99.0,88.0,107.0,122.0,...,58.0,90.0,76.0,71.0,75.0,93.0,74.0,94.0,9.0,NaN
2010-02,376.0,89.0,84.0,109.0,92.0,75.0,72.0,107.0,95.0,103.0,...,75.0,60.0,61.0,54.0,86.0,86.0,61.0,10.0,NaN,NaN
2010-03,443.0,84.0,102.0,107.0,103.0,90.0,109.0,134.0,122.0,48.0,...,75.0,77.0,69.0,78.0,89.0,94.0,11.0,NaN,NaN,NaN
2010-04,294.0,57.0,57.0,48.0,54.0,66.0,81.0,77.0,31.0,32.0,...,46.0,41.0,44.0,53.0,66.0,5.0,NaN,NaN,NaN,NaN
2010-05,254.0,40.0,43.0,44.0,45.0,65.0,54.0,32.0,15.0,21.0,...,32.0,35.0,42.0,39.0,5.0,NaN,NaN,NaN,NaN,NaN
2010-06,270.0,47.0,51.0,55.0,62.0,77.0,34.0,24.0,22.0,32.0,...,33.0,36.0,55.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,186.0,29.0,34.0,55.0,54.0,26.0,21.0,27.0,27.0,21.0,...,32.0,44.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,162.0,33.0,48.0,52.0,28.0,19.0,16.0,20.0,22.0,21.0,...,32.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
retention_matrix

CohortIndex,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,25
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2009-12,1.0,0.352880,0.334031,0.425131,0.380105,0.359162,0.376963,0.342408,0.336126,0.362304,...,0.302618,0.262827,0.302618,0.282723,0.259686,0.255497,0.315183,0.304712,0.407330,0.072251
2010-01,1.0,0.206266,0.310705,0.305483,0.263708,0.300261,0.258486,0.229765,0.279373,0.318538,...,0.151436,0.234987,0.198433,0.185379,0.195822,0.242820,0.193211,0.245431,0.023499,NaN
2010-02,1.0,0.236702,0.223404,0.289894,0.244681,0.199468,0.191489,0.284574,0.252660,0.273936,...,0.199468,0.159574,0.162234,0.143617,0.228723,0.228723,0.162234,0.026596,NaN,NaN
2010-03,1.0,0.189616,0.230248,0.241535,0.232506,0.203160,0.246050,0.302483,0.275395,0.108352,...,0.169300,0.173815,0.155756,0.176072,0.200903,0.212190,0.024831,NaN,NaN,NaN
2010-04,1.0,0.193878,0.193878,0.163265,0.183673,0.224490,0.275510,0.261905,0.105442,0.108844,...,0.156463,0.139456,0.149660,0.180272,0.224490,0.017007,NaN,NaN,NaN,NaN
2010-05,1.0,0.157480,0.169291,0.173228,0.177165,0.255906,0.212598,0.125984,0.059055,0.082677,...,0.125984,0.137795,0.165354,0.153543,0.019685,NaN,NaN,NaN,NaN,NaN
2010-06,1.0,0.174074,0.188889,0.203704,0.229630,0.285185,0.125926,0.088889,0.081481,0.118519,...,0.122222,0.133333,0.203704,0.018519,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,1.0,0.155914,0.182796,0.295699,0.290323,0.139785,0.112903,0.145161,0.145161,0.112903,...,0.172043,0.236559,0.032258,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,1.0,0.203704,0.296296,0.320988,0.172840,0.117284,0.098765,0.123457,0.135802,0.129630,...,0.197531,0.037037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Step 3: RFM Calculations

In [22]:
import datetime as dt

In [23]:
latest_date=df['InvoiceDate'].max()+dt.timedelta(days=1)

In [24]:
rfm=df.groupby('Customer ID').agg({
    'InvoiceDate':lambda x:(latest_date-x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice':'sum'
    
})

In [26]:
rfm.columns=['Recency','Frequency','Monetary']

### Using pd.qcut to assign scores from 1 to 5

In [35]:
rfm['R_Score']=pd.qcut(rfm['Recency'].rank(method='first'),5,labels=[1,2,3,4,5])

In [36]:
rfm['F_Score']=pd.qcut(rfm['Frequency'].rank(method='first'),5,labels=[1,2,3,4,5])

In [37]:
rfm['M_Score']=pd.qcut(rfm['Monetary'].rank(method='first'),5,labels=[1,2,3,4,5])

In [40]:
rfm.sample()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Group
Customer ID,,,,,,,
17875.0,683,1,167.1,5,2,1,521


### Creating a combined RFM Score 

In [39]:
rfm['RFM_Group']=rfm.R_Score.astype(str)+rfm.F_Score.astype(str)+rfm.M_Score.astype(str)

In [41]:
rfm

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Group
Customer ID,,,,,,,
12346.0,321,12,77556.46,4,5,5,455
12347.0,35,7,5408.50,2,4,5,245
12348.0,71,5,2019.40,3,4,4,344
12349.0,14,4,4428.69,1,3,5,135
12350.0,305,1,334.40,4,1,2,412
...,...,...,...,...,...,...,...
18283.0,5,21,2528.65,1,5,4,154
18284.0,427,1,461.68,5,2,2,522
18285.0,656,1,427.00,5,2,2,522


### exporting the dataframe as csv file

In [43]:
rfm.to_csv('rfmanalysis.csv', index=True)